# DOE MASTER FULL PIPELINE — original workbooks → WellA–WellD

This notebook replaces the older curated-workbook workflow. It rebuilds four standardized well tables directly from the three **original** Excel workbooks and then runs leakage-safe cross-well saturation modeling.

## Authoritative source map

| Output alias | Original workbook | Sheet rule |
|---|---|---|
| `WellA` | `2L-38_input_Mallik.xlsx` | best recognized non-refined log sheet |
| `WellB` | `5L-38_input_Mallik.xlsx` | best recognized non-refined log sheet |
| `WellC` | `MtElbert_Ignik_input_ANS.xlsx` | `MTE` |
| `WellD` | `MtElbert_Ignik_input_ANS.xlsx` | `IGS` |

The notebook explicitly ignores every `curated_dataset*.xlsx` file. `MTE_refined` and `IGS_refined` are inventoried as alignment/QC references only; they are not counted as additional wells and are never used as training tables.

## What normalization means

- Source headers are mapped to canonical names.
- Physical units are standardized while source provenance is retained.
- Original depth is preserved and `depth_m` is added; depth is never min–max normalized or used as a predictor by default.
- Porosity and saturation are converted to fractions only when the source is clearly percent-scaled.
- `Sgh`, `S_h`, `Sh`, and `NMR_SAT` map to `hydrate_saturation_vv`.
- `S_wr` and `Swr` map separately to `water_saturation_vv`.
- Model imputation and 0–1 feature scaling are fitted only on training wells inside each split.
- NMR porosity is excluded from predictors by default because the Mallik saturation label is described as NMR-density-derived; this avoids circular target reconstruction unless provenance review later authorizes it.

## Validation design

1. **Single-source transfer:** train on `WellC` by default and predict `WellA`, `WellB`, and `WellD`.
2. **Leave-one-well-out:** train on the other labeled wells and evaluate each completely held-out well.
3. Hydrate and water saturation are modeled separately. Water-saturation metrics are produced only where reference values exist.

Run the notebook from top to bottom. It hard-stops if an original workbook is missing, depth appears normalized, targets remain outside the expected fraction range, or fewer than three shared predictors are available.

## 0. Imports, exact filenames, and runtime configuration

The default input folder is `Downloads/Northslopedatasets06052026`. Set the `NORTH_SLOPE_DATA_DIR` environment variable only when the folder is elsewhere. The physical-unit standardized tables remain local under `outputs_runtime/`; fitted models remain under `models_runtime/`.

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable
import json
import math
import os
import re
import shutil
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

# =============================================================================
# Configuration
# =============================================================================

DEFAULT_DATA_DIR = Path.home() / "Downloads" / "Northslopedatasets06052026"
DATA_DIR = Path(os.environ.get("NORTH_SLOPE_DATA_DIR", DEFAULT_DATA_DIR)).expanduser()

SOURCE_SPECS: list[dict[str, Any]] = [
    {
        "well_alias": "WellA",
        "filename": "2L-38_input_Mallik.xlsx",
        "sheet_name": None,
        "source_label": "Mallik 2L-38",
        "default_depth_unit": "m",
        "default_density_unit": "kg/m3",
        "default_caliper_unit": "mm",
    },
    {
        "well_alias": "WellB",
        "filename": "5L-38_input_Mallik.xlsx",
        "sheet_name": None,
        "source_label": "Mallik 5L-38",
        "default_depth_unit": "m",
        "default_density_unit": "kg/m3",
        "default_caliper_unit": "mm",
    },
    {
        "well_alias": "WellC",
        "filename": "MtElbert_Ignik_input_ANS.xlsx",
        "sheet_name": "MTE",
        "source_label": "MTE",
        "default_depth_unit": "ft",
        "default_density_unit": "g/cc",
        "default_caliper_unit": "in",
    },
    {
        "well_alias": "WellD",
        "filename": "MtElbert_Ignik_input_ANS.xlsx",
        "sheet_name": "IGS",
        "source_label": "IGS",
        "default_depth_unit": "ft",
        "default_density_unit": "g/cc",
        "default_caliper_unit": "in",
    },
]

CURATED_GLOB = "curated_dataset*.xlsx"
REFINED_SHEETS = ("MTE_refined", "IGS_refined")

OUTPUT_ROOT = Path.cwd() / "outputs_runtime"
MODEL_ROOT = Path.cwd() / "models_runtime"
RUN_ROOT = OUTPUT_ROOT / "original_workbook_pipeline"
STANDARDIZED_ROOT = RUN_ROOT / "standardized_wells"
AUDIT_ROOT = RUN_ROOT / "audits"
PREDICTION_ROOT = RUN_ROOT / "predictions"
METRIC_ROOT = RUN_ROOT / "metrics"
PAPER_EXPORT_ROOT = OUTPUT_ROOT / "paper_slide_model_exports"
MODEL_RUN_ROOT = MODEL_ROOT / "original_workbook_pipeline"
VALIDATION_BUNDLE_ROOT = Path.cwd() / "north_slope_validation_bundle"

OVERWRITE_RUN = True
SINGLE_TRAIN_WELL = "WellC"
MODEL_KIND = "random_forest"  # random_forest or mlp
RANDOM_STATE = 42
MIN_TARGET_ROWS = 20
MIN_FEATURE_COVERAGE = 0.20
USE_DEPTH_AS_FEATURE = False
ALLOW_PROVISIONAL_A090_AS_RESISTIVITY = True

# Preserve physical-unit standardized tables. Model-only scaling is fitted inside
# each training fold and never uses held-out well rows.
SCALE_MODEL_FEATURES = True
CLIP_SATURATION_PREDICTIONS = True

# Mallik hydrate saturation is labeled as NMR-density-derived in the workbook.
# Keep NMR porosity and NMR-density separation out of predictors by default to
# avoid circular target reconstruction. Set True only after provenance review.
ALLOW_NMR_POROSITY_AS_FEATURE = False

TARGET_COLUMNS = ("hydrate_saturation_vv", "water_saturation_vv")

## 1. Canonical fields and leakage policy

This registry supports the stacked Mallik role/mnemonic/unit/description layout and the direct MTE/IGS headers. Target aliases remain target-only and are never inserted into the feature matrix.

In [ ]:
# =============================================================================
# Canonical schema
# =============================================================================

def normalize_token(value: Any) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return "".join(character for character in str(value).strip().lower() if character.isalnum())


ALIASES: dict[str, tuple[str, ...]] = {
    "depth": (
        "depth", "depth_ft", "depth, ft", "depth ft", "depth m", "depth_m", "dept",
        "true depth", "measured depth", "md", "tvd",
    ),
    "rhob_g_cc": (
        "rho_b", "rhob", "density_gpcc", "density_gcpcc", "density g/cc", "density gpcc",
        "bulk density", "density",
    ),
    "density_porosity_vv": (
        "phi_porosity", "dphi", "phi_den", "density porosity", "density_porosity",
    ),
    "neutron_porosity_vv": ("nphi", "phi_neut", "neutron porosity"),
    "nmr_porosity_vv": ("nmrphi", "phi_nmr", "nmr porosity"),
    "gr_api": ("gr", "gamma ray", "gamma_ray"),
    "caliper_in": ("caliper", "cal1", "cali"),
    "differential_caliper_mm": ("differential caliper", "differential_caliper", "diff caliper"),
    "rt_ohm_m": (
        "res", "rt", "deep formation resistivity", "apparent resistivity", "a090", "af90",
        "computed focusing mode 5", "resistivity",
    ),
    "vp_m_s": ("vp", "velp", "compressional wave velocity", "compressional velocity"),
    "vs_m_s": ("vs", "vs1", "shear wave velocity", "shear velocity"),
    "vp_vs_ratio": ("ratio vp/vs", "ratio vpvs", "vp/vs", "vp_vs_ratio", "ratio of velocities"),
    "acoustic_impedance_source": ("impedance", "acoustic impedance", "rho_b*vp", "rhob*vp"),
    "hydrate_saturation_vv": (
        "sgh", "s_h", "sh", "hydrate saturation", "hydrate_saturation", "hydrate sat",
        "nmr_sat", "gas hydrate saturation",
    ),
    "water_saturation_vv": (
        "s_wr", "swr", "swirr", "irreducible water saturation", "residual water saturation",
    ),
    "alignment_depth_unit_d": ("depths_unitd", "unit d depth", "unitd depth"),
    "alignment_depth_unit_c": ("depths_unitc", "unit c depth", "unitc depth"),
    "aligned_depth": ("depth correspondence at ml data", "aligned depth", "depth correspondence"),
}

ALIAS_LOOKUP: dict[str, str] = {}
for canonical_name, source_aliases in ALIASES.items():
    for alias in (canonical_name, *source_aliases):
        ALIAS_LOOKUP[normalize_token(alias)] = canonical_name

TARGET_ALIAS_TOKENS = {
    token for token, canonical in ALIAS_LOOKUP.items()
    if canonical in TARGET_COLUMNS
}

MODEL_FEATURE_CANDIDATES = (
    "rhob_g_cc",
    "density_porosity_vv",
    "neutron_porosity_vv",
    "gr_api",
    "caliper_in",
    "differential_caliper_mm",
    "rt_ohm_m",
    "vp_m_s",
    "vs_m_s",
    "vp_vs_ratio",
    "acoustic_impedance_kg_m2_s",
    "log10_rt_ohm_m",
)
if ALLOW_NMR_POROSITY_AS_FEATURE:
    MODEL_FEATURE_CANDIDATES = MODEL_FEATURE_CANDIDATES + (
        "nmr_porosity_vv",
        "nmr_density_separation_vv",
    )

ALIGNMENT_CANONICAL_COLUMNS = {
    "alignment_depth_unit_d", "alignment_depth_unit_c", "aligned_depth"
}

ROLE_TOKENS = {"mlinput", "groundtruth", "outlierremoval", "depth", "qc", "target"}


def canonical_for_header(value: Any) -> str | None:
    token = normalize_token(value)
    canonical = ALIAS_LOOKUP.get(token)
    if token in {"a090", "af90"} and not ALLOW_PROVISIONAL_A090_AS_RESISTIVITY:
        return None
    return canonical


def clean_label(value: Any, fallback: str) -> str:
    try:
        missing = pd.isna(value)
    except Exception:
        missing = False
    text = "" if missing else str(value).strip()
    text = re.sub(r"\s+", " ", text)
    return text or fallback


def dedupe_labels(labels: Iterable[str]) -> list[str]:
    seen: dict[str, int] = {}
    result: list[str] = []
    for label in labels:
        count = seen.get(label, 0)
        seen[label] = count + 1
        result.append(label if count == 0 else f"{label}__{count + 1}")
    return result


def numeric_series(values: pd.Series) -> pd.Series:
    cleaned = values.astype(str).str.replace(",", "", regex=False).str.replace("%", "", regex=False)
    cleaned = cleaned.replace({"": np.nan, "nan": np.nan, "None": np.nan, "-": np.nan})
    return pd.to_numeric(cleaned, errors="coerce").replace([np.inf, -np.inf], np.nan)

## 2. Workbook parsing and unit-aware standardization

The parser detects the real mnemonic row, locates the first numeric data row, records source role/unit/description metadata, preserves original depth, and converts only documented or strongly inferred units.

In [ ]:
# =============================================================================
# Workbook parsing
# =============================================================================

def header_row_score(row: pd.Series) -> tuple[int, int, int]:
    tokens = [normalize_token(value) for value in row.tolist() if normalize_token(value)]
    exact = sum(token in ALIAS_LOOKUP for token in tokens)
    target = sum(token in TARGET_ALIAS_TOKENS for token in tokens)
    keyword = sum(
        any(part in token for part in (
            "depth", "density", "porosity", "caliper", "resist", "gamma", "velocity",
            "impedance", "saturation",
        ))
        for token in tokens
    )
    return exact * 10 + target * 3 + keyword, exact, len(tokens)


def detect_header_layout(raw: pd.DataFrame, max_scan_rows: int = 10) -> dict[str, Any]:
    if raw.empty:
        raise ValueError("Sheet is empty.")
    scan_count = min(max_scan_rows, len(raw))
    scores = [header_row_score(raw.iloc[row_index]) for row_index in range(scan_count)]
    header_index = max(range(scan_count), key=lambda index: scores[index])
    if scores[header_index][1] < 3:
        raise ValueError(f"Could not identify a reliable mnemonic header row. Scores: {scores}")

    header_values = [
        clean_label(value, f"unnamed_{column_index}")
        for column_index, value in enumerate(raw.iloc[header_index].tolist())
    ]
    meaningful_columns = [index for index, label in enumerate(header_values) if not label.startswith("unnamed_")]

    data_start = None
    for row_index in range(header_index + 1, min(len(raw), header_index + 12)):
        row = raw.iloc[row_index, meaningful_columns] if meaningful_columns else raw.iloc[row_index]
        nonempty = int(row.notna().sum())
        numeric = int(numeric_series(row).notna().sum())
        if numeric >= 3 and numeric / max(nonempty, 1) >= 0.45:
            data_start = row_index
            break
    if data_start is None:
        raise ValueError("Could not identify the first numeric data row after the header.")

    role_index = None
    if header_index > 0:
        role_tokens = {normalize_token(value) for value in raw.iloc[header_index - 1].tolist()}
        if role_tokens & ROLE_TOKENS:
            role_index = header_index - 1

    unit_index = None
    description_index = None
    for row_index in range(header_index + 1, data_start):
        tokens = [normalize_token(value) for value in raw.iloc[row_index].tolist() if normalize_token(value)]
        unit_hits = sum(
            token in {"m", "ft", "kgm3", "gcc", "gcm3", "api", "mm", "in", "ohmm", "ms", "kms", "ratio", "nmrsat"}
            or any(unit in token for unit in ("kgm3", "ohmm", "ms", "kms", "api", "mm", "inch", "feet"))
            for token in tokens
        )
        if unit_hits >= 2 and unit_index is None:
            unit_index = row_index
        description_index = row_index

    return {
        "header_index": header_index,
        "data_start": data_start,
        "role_index": role_index,
        "unit_index": unit_index,
        "description_index": description_index,
        "header_score": scores[header_index][0],
        "recognized_headers": scores[header_index][1],
    }


def choose_best_sheet(path: Path) -> tuple[str, pd.DataFrame, dict[str, Any]]:
    with pd.ExcelFile(path) as excel:
        sheet_names = list(excel.sheet_names)
    candidates: list[tuple[int, int, str, pd.DataFrame, dict[str, Any]]] = []
    errors: list[str] = []
    for sheet_name in sheet_names:
        if "refined" in normalize_token(sheet_name):
            continue
        try:
            raw = pd.read_excel(path, sheet_name=sheet_name, header=None)
            layout = detect_header_layout(raw)
            candidates.append((layout["recognized_headers"], len(raw), sheet_name, raw, layout))
        except Exception as exc:
            errors.append(f"{sheet_name}: {exc}")
    if not candidates:
        raise ValueError(f"No model-input sheet could be identified in {path.name}. Details: {errors}")
    _, _, sheet_name, raw, layout = sorted(candidates, key=lambda item: (item[0], item[1]), reverse=True)[0]
    return sheet_name, raw, layout


def read_source_sheet(path: Path, requested_sheet: str | None) -> tuple[str, pd.DataFrame, dict[str, Any]]:
    if requested_sheet is None:
        return choose_best_sheet(path)
    with pd.ExcelFile(path) as excel:
        matching = {normalize_token(name): name for name in excel.sheet_names}
    actual = matching.get(normalize_token(requested_sheet))
    if actual is None:
        raise ValueError(f"Sheet {requested_sheet!r} is missing from {path.name}.")
    raw = pd.read_excel(path, sheet_name=actual, header=None)
    return actual, raw, detect_header_layout(raw)


def infer_unit(header: str, unit: str, default: str, canonical: str) -> str:
    combined = f"{header} {unit}".lower().replace("³", "3")
    token = normalize_token(combined)
    if canonical == "depth":
        if "ft" in combined or "feet" in combined:
            return "ft"
        if re.search(r"(^|\s)m($|\s)", combined) and "mm" not in combined and "/s" not in combined:
            return "m"
    if canonical == "rhob_g_cc":
        if "kg/m3" in combined or "kgm3" in token:
            return "kg/m3"
        if any(value in combined for value in ("g/cc", "g/cm3", "gpcc", "gcpcc")):
            return "g/cc"
    if canonical in {"caliper_in", "differential_caliper_mm"}:
        if "mm" in combined:
            return "mm"
        if any(value in combined for value in ("inch", " in", "in.")):
            return "in"
    if canonical in {"vp_m_s", "vs_m_s"}:
        if "km/s" in combined or "kms" in token:
            return "km/s"
        if "m/s" in combined or token.endswith("ms"):
            return "m/s"
    return default


def convert_fraction(
    values: pd.Series,
    field: str,
    audit: list[dict[str, Any]],
    well_alias: str,
    source_header: str,
) -> pd.Series:
    numeric = numeric_series(values)
    finite = numeric.dropna()
    conversion = "none"
    if not finite.empty:
        q99 = float(finite.quantile(0.99))
        if q99 > 1.5 and q99 <= 100.5:
            numeric = numeric / 100.0
            conversion = "percent_to_fraction"
    audit.append({
        "well_alias": well_alias,
        "field": field,
        "source_header": source_header,
        "source_unit": "unknown_or_fraction",
        "canonical_unit": "fraction",
        "conversion": conversion,
        "source_min": float(finite.min()) if not finite.empty else np.nan,
        "source_max": float(finite.max()) if not finite.empty else np.nan,
    })
    return numeric


def standardize_well(spec: dict[str, Any], data_dir: Path | None = None) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    data_dir = Path(data_dir or DATA_DIR)
    well_alias = spec["well_alias"]
    path = data_dir / spec["filename"]
    if not path.exists():
        raise FileNotFoundError(path)
    sheet_name, raw, layout = read_source_sheet(path, spec.get("sheet_name"))

    headers = dedupe_labels([
        clean_label(value, f"unnamed_{column_index}")
        for column_index, value in enumerate(raw.iloc[layout["header_index"]].tolist())
    ])
    roles = (
        [clean_label(value, "") for value in raw.iloc[layout["role_index"]].tolist()]
        if layout["role_index"] is not None else [""] * len(headers)
    )
    units = (
        [clean_label(value, "") for value in raw.iloc[layout["unit_index"]].tolist()]
        if layout["unit_index"] is not None else [""] * len(headers)
    )
    descriptions = (
        [clean_label(value, "") for value in raw.iloc[layout["description_index"]].tolist()]
        if layout["description_index"] is not None else [""] * len(headers)
    )

    data = raw.iloc[layout["data_start"]:].copy().reset_index(drop=True)
    data.columns = headers
    data = data.dropna(axis=0, how="all").dropna(axis=1, how="all")

    mapping_rows: list[dict[str, Any]] = []
    unit_audit: list[dict[str, Any]] = []
    canonical_sources: dict[str, str] = {}
    canonical_series: dict[str, pd.Series] = {}

    for column_index, header in enumerate(headers):
        if header not in data.columns:
            continue
        canonical = canonical_for_header(header)
        role = roles[column_index] if column_index < len(roles) else ""
        unit = units[column_index] if column_index < len(units) else ""
        description = descriptions[column_index] if column_index < len(descriptions) else ""
        mapping_rows.append({
            "well_alias": well_alias,
            "source_workbook": spec["filename"],
            "source_sheet": sheet_name,
            "column_position": column_index + 1,
            "source_header": header,
            "source_role": role,
            "source_unit": unit,
            "source_description": description,
            "canonical_field": canonical or "unmapped",
            "model_permission": (
                "target_only" if canonical in TARGET_COLUMNS else
                "alignment_only" if canonical in ALIGNMENT_CANONICAL_COLUMNS else
                "candidate_feature" if canonical in MODEL_FEATURE_CANDIDATES or canonical in {
                    "rhob_g_cc", "density_porosity_vv", "neutron_porosity_vv", "nmr_porosity_vv",
                    "gr_api", "caliper_in", "differential_caliper_mm", "rt_ohm_m", "vp_m_s",
                    "vs_m_s", "vp_vs_ratio", "acoustic_impedance_source",
                } else "excluded_or_unmapped"
            ),
        })
        if canonical is None or canonical in ALIGNMENT_CANONICAL_COLUMNS:
            continue
        candidate = numeric_series(data[header])
        if canonical in canonical_series:
            if candidate.notna().sum() > canonical_series[canonical].notna().sum():
                canonical_series[canonical] = candidate
                canonical_sources[canonical] = header
        else:
            canonical_series[canonical] = candidate
            canonical_sources[canonical] = header

    standardized = pd.DataFrame(index=data.index)
    standardized["well_alias"] = well_alias
    standardized["source_label"] = spec["source_label"]
    standardized["source_workbook"] = spec["filename"]
    standardized["source_sheet"] = sheet_name
    standardized["source_row"] = np.arange(layout["data_start"] + 1, layout["data_start"] + 1 + len(data))

    if "depth" not in canonical_series:
        raise ValueError(f"No depth column was mapped for {well_alias} from {path.name}/{sheet_name}.")

    depth_source_header = canonical_sources["depth"]
    depth_column_position = headers.index(depth_source_header)
    depth_unit_text = units[depth_column_position] if depth_column_position < len(units) else ""
    depth_unit = infer_unit(depth_source_header, depth_unit_text, spec["default_depth_unit"], "depth")
    depth_original = numeric_series(data[depth_source_header])
    standardized["depth_original"] = depth_original
    standardized["depth_original_unit"] = depth_unit
    standardized["depth_m"] = depth_original * 0.3048 if depth_unit == "ft" else depth_original
    unit_audit.append({
        "well_alias": well_alias,
        "field": "depth_m",
        "source_header": depth_source_header,
        "source_unit": depth_unit,
        "canonical_unit": "m",
        "conversion": "multiply_0.3048" if depth_unit == "ft" else "identity",
        "source_min": float(depth_original.min()) if depth_original.notna().any() else np.nan,
        "source_max": float(depth_original.max()) if depth_original.notna().any() else np.nan,
    })

    standard_fields = (
        "rhob_g_cc", "density_porosity_vv", "neutron_porosity_vv", "nmr_porosity_vv",
        "gr_api", "caliper_in", "differential_caliper_mm", "rt_ohm_m", "vp_m_s", "vs_m_s",
        "vp_vs_ratio", "acoustic_impedance_source", "hydrate_saturation_vv", "water_saturation_vv",
    )
    for field in standard_fields:
        if field not in canonical_series:
            continue
        source_header = canonical_sources[field]
        source_position = headers.index(source_header)
        source_unit_text = units[source_position] if source_position < len(units) else ""
        values = canonical_series[field].copy()

        if field == "rhob_g_cc":
            unit = infer_unit(source_header, source_unit_text, spec["default_density_unit"], field)
            finite = values.dropna()
            inferred_by_magnitude = False
            if unit == "g/cc" and not finite.empty and float(finite.median()) > 20:
                unit = "kg/m3"
                inferred_by_magnitude = True
            standardized[field] = values / 1000.0 if unit == "kg/m3" else values
            unit_audit.append({
                "well_alias": well_alias, "field": field, "source_header": source_header,
                "source_unit": unit, "canonical_unit": "g/cc",
                "conversion": "divide_1000" if unit == "kg/m3" else "identity",
                "inferred_by_magnitude": inferred_by_magnitude,
            })
        elif field in {"density_porosity_vv", "neutron_porosity_vv", "nmr_porosity_vv", *TARGET_COLUMNS}:
            if field in TARGET_COLUMNS:
                standardized[f"{field}_original"] = values
                standardized[f"{field}_source_header"] = source_header
            standardized[field] = convert_fraction(values, field, unit_audit, well_alias, source_header)
        elif field == "caliper_in":
            unit = infer_unit(source_header, source_unit_text, spec["default_caliper_unit"], field)
            finite = values.dropna()
            if unit == "in" and not finite.empty and float(finite.median()) > 50:
                unit = "mm"
            standardized[field] = values / 25.4 if unit == "mm" else values
            unit_audit.append({
                "well_alias": well_alias, "field": field, "source_header": source_header,
                "source_unit": unit, "canonical_unit": "in",
                "conversion": "divide_25.4" if unit == "mm" else "identity",
            })
        elif field == "differential_caliper_mm":
            unit = infer_unit(source_header, source_unit_text, "mm", field)
            standardized[field] = values * 25.4 if unit == "in" else values
            unit_audit.append({
                "well_alias": well_alias, "field": field, "source_header": source_header,
                "source_unit": unit, "canonical_unit": "mm",
                "conversion": "multiply_25.4" if unit == "in" else "identity",
            })
        elif field in {"vp_m_s", "vs_m_s"}:
            unit = infer_unit(source_header, source_unit_text, "m/s", field)
            finite = values.dropna()
            if unit == "m/s" and not finite.empty and 0 < float(finite.median()) < 20:
                unit = "km/s"
            standardized[field] = values * 1000.0 if unit == "km/s" else values
            unit_audit.append({
                "well_alias": well_alias, "field": field, "source_header": source_header,
                "source_unit": unit, "canonical_unit": "m/s",
                "conversion": "multiply_1000" if unit == "km/s" else "identity",
            })
        else:
            standardized[field] = values
            unit_audit.append({
                "well_alias": well_alias, "field": field, "source_header": source_header,
                "source_unit": source_unit_text or "unknown", "canonical_unit": "preserved",
                "conversion": "identity",
            })

    if {"vp_m_s", "vs_m_s"}.issubset(standardized.columns):
        ratio = standardized["vp_m_s"] / standardized["vs_m_s"].replace(0, np.nan)
        if "vp_vs_ratio" not in standardized or standardized["vp_vs_ratio"].notna().mean() < 0.20:
            standardized["vp_vs_ratio"] = ratio
        else:
            standardized["vp_vs_ratio_recomputed"] = ratio
    if {"rhob_g_cc", "vp_m_s"}.issubset(standardized.columns):
        standardized["acoustic_impedance_kg_m2_s"] = standardized["rhob_g_cc"] * 1000.0 * standardized["vp_m_s"]
    if "rt_ohm_m" in standardized:
        positive_rt = standardized["rt_ohm_m"].where(standardized["rt_ohm_m"] > 0)
        standardized["log10_rt_ohm_m"] = np.log10(positive_rt)
    if {"density_porosity_vv", "nmr_porosity_vv"}.issubset(standardized.columns):
        standardized["nmr_density_separation_vv"] = standardized["density_porosity_vv"] - standardized["nmr_porosity_vv"]

    numeric_columns = [column for column in standardized.columns if column not in {
        "well_alias", "source_label", "source_workbook", "source_sheet", "depth_original_unit",
        "hydrate_saturation_vv_source_header", "water_saturation_vv_source_header",
    }]
    for column in numeric_columns:
        standardized[column] = pd.to_numeric(standardized[column], errors="coerce").replace([np.inf, -np.inf], np.nan)

    signal_columns = [column for column in standardized.columns if column in MODEL_FEATURE_CANDIDATES or column in TARGET_COLUMNS]
    keep_mask = standardized["depth_m"].notna()
    if signal_columns:
        keep_mask &= standardized[signal_columns].notna().any(axis=1)
    standardized = standardized.loc[keep_mask].copy()
    standardized = standardized.sort_values(["depth_m", "source_row"], kind="stable").reset_index(drop=True)

    layout_record = {
        "well_alias": well_alias,
        "source_workbook": spec["filename"],
        "source_sheet": sheet_name,
        "header_row_excel": layout["header_index"] + 1,
        "data_start_row_excel": layout["data_start"] + 1,
        "role_row_excel": layout["role_index"] + 1 if layout["role_index"] is not None else None,
        "unit_row_excel": layout["unit_index"] + 1 if layout["unit_index"] is not None else None,
        "description_row_excel": layout["description_index"] + 1 if layout["description_index"] is not None else None,
        "recognized_header_count": layout["recognized_headers"],
        "standardized_rows": len(standardized),
    }
    return standardized, pd.DataFrame(mapping_rows), pd.DataFrame(unit_audit), layout_record


def refined_sheet_inventory(path: Path) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    if not path.exists():
        return pd.DataFrame(rows)
    with pd.ExcelFile(path) as excel:
        names = {normalize_token(name): name for name in excel.sheet_names}
    for requested in REFINED_SHEETS:
        actual = names.get(normalize_token(requested))
        if actual is None:
            rows.append({"source_workbook": path.name, "source_sheet": requested, "status": "missing"})
            continue
        raw = pd.read_excel(path, sheet_name=actual, header=None)
        target_like_cells = sum(
            normalize_token(value) in TARGET_ALIAS_TOKENS
            for value in raw.head(12).to_numpy().ravel().tolist()
        )
        rows.append({
            "source_workbook": path.name,
            "source_sheet": actual,
            "status": "excluded_alignment_qc_only",
            "rows": len(raw),
            "columns": len(raw.columns),
            "target_like_header_cells_first_12_rows": target_like_cells,
            "reason": "Refined/alignment sheets are not counted as wells and are never used as training tables.",
        })
    return pd.DataFrame(rows)

## 3. Standardized WellA–WellD tables and hard-stop audits

This section writes the four well tables plus source-layout, column-mapping, unit-conversion, target-range, feature-coverage, refined-sheet, ignored-file, and target-leakage audits.

In [ ]:
# =============================================================================
# Standardization and audits
# =============================================================================

def prepare_output_directories() -> None:
    if OVERWRITE_RUN:
        for path in (RUN_ROOT, MODEL_RUN_ROOT, PAPER_EXPORT_ROOT, VALIDATION_BUNDLE_ROOT):
            if path.exists():
                shutil.rmtree(path)
    for path in (
        STANDARDIZED_ROOT, AUDIT_ROOT, PREDICTION_ROOT, METRIC_ROOT,
        PAPER_EXPORT_ROOT, MODEL_RUN_ROOT,
    ):
        path.mkdir(parents=True, exist_ok=True)


def standardize_all_sources(data_dir: Path | None = None) -> tuple[dict[str, pd.DataFrame], dict[str, pd.DataFrame]]:
    data_dir = Path(data_dir or DATA_DIR)
    prepare_output_directories()
    missing = [spec["filename"] for spec in SOURCE_SPECS if not (data_dir / spec["filename"]).exists()]
    if missing:
        raise FileNotFoundError(
            "Missing original workbooks in " + str(data_dir) + ": " + ", ".join(missing)
        )

    ignored_files = sorted(path.name for path in data_dir.glob(CURATED_GLOB))
    ignored_df = pd.DataFrame([
        {"filename": name, "status": "ignored", "reason": "Curated/normalized duplicate; original workbook is authoritative."}
        for name in ignored_files
    ])
    ignored_df.to_csv(AUDIT_ROOT / "ignored_input_inventory.csv", index=False)

    well_frames: dict[str, pd.DataFrame] = {}
    mappings: list[pd.DataFrame] = []
    units: list[pd.DataFrame] = []
    layouts: list[dict[str, Any]] = []
    for spec in SOURCE_SPECS:
        frame, mapping, unit_audit, layout = standardize_well(spec, data_dir=data_dir)
        well_alias = spec["well_alias"]
        well_frames[well_alias] = frame
        mappings.append(mapping)
        units.append(unit_audit)
        layouts.append(layout)
        well_dir = STANDARDIZED_ROOT / well_alias
        well_dir.mkdir(parents=True, exist_ok=True)
        frame.to_csv(well_dir / f"{well_alias}_standardized.csv", index=False)

    combined = pd.concat(well_frames.values(), ignore_index=True, sort=False)
    combined.to_csv(STANDARDIZED_ROOT / "WellA_WellB_WellC_WellD_standardized.csv", index=False)
    mapping_df = pd.concat(mappings, ignore_index=True, sort=False)
    unit_df = pd.concat(units, ignore_index=True, sort=False)
    layout_df = pd.DataFrame(layouts)
    mapping_df.to_csv(AUDIT_ROOT / "column_mapping_audit.csv", index=False)
    unit_df.to_csv(AUDIT_ROOT / "unit_conversion_audit.csv", index=False)
    layout_df.to_csv(AUDIT_ROOT / "source_layout_inventory.csv", index=False)

    refined = refined_sheet_inventory(data_dir / "MtElbert_Ignik_input_ANS.xlsx")
    refined.to_csv(AUDIT_ROOT / "refined_sheet_inventory.csv", index=False)

    return well_frames, {
        "combined": combined,
        "column_mapping": mapping_df,
        "unit_conversion": unit_df,
        "source_layout": layout_df,
        "refined_sheet_inventory": refined,
        "ignored_inputs": ignored_df,
    }


def build_target_and_readiness_audits(
    well_frames: dict[str, pd.DataFrame],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    target_rows: list[dict[str, Any]] = []
    readiness_rows: list[dict[str, Any]] = []
    feature_rows: list[dict[str, Any]] = []
    blocking_messages: list[str] = []

    for well_alias, frame in well_frames.items():
        depth = frame["depth_m"].dropna()
        suspicious_depth = depth.empty or float(depth.max()) <= 10 or float(depth.max() - depth.min()) <= 1
        if suspicious_depth:
            blocking_messages.append(f"{well_alias}: depth appears normalized or invalid.")
        readiness_rows.append({
            "well_alias": well_alias,
            "rows": len(frame),
            "depth_non_null": int(frame["depth_m"].notna().sum()),
            "depth_min_m": float(depth.min()) if not depth.empty else np.nan,
            "depth_max_m": float(depth.max()) if not depth.empty else np.nan,
            "depth_span_m": float(depth.max() - depth.min()) if not depth.empty else np.nan,
            "depth_monotonic": bool(depth.is_monotonic_increasing),
            "depth_duplicate_rows": int(frame["depth_m"].duplicated().sum()),
            "suspicious_normalized_depth": suspicious_depth,
        })
        for target in TARGET_COLUMNS:
            series = frame[target] if target in frame else pd.Series(index=frame.index, dtype=float)
            valid = series.dropna()
            out_of_range = int(((valid < -0.001) | (valid > 1.001)).sum()) if not valid.empty else 0
            constant = bool(valid.nunique() <= 1) if not valid.empty else False
            eligible = bool(valid.size >= MIN_TARGET_ROWS and not constant and out_of_range == 0)
            target_rows.append({
                "well_alias": well_alias,
                "target": target,
                "rows": len(frame),
                "target_rows": int(valid.size),
                "target_fraction": float(valid.size / max(len(frame), 1)),
                "target_min": float(valid.min()) if not valid.empty else np.nan,
                "target_max": float(valid.max()) if not valid.empty else np.nan,
                "target_mean": float(valid.mean()) if not valid.empty else np.nan,
                "unique_values": int(valid.nunique()),
                "constant_target": constant,
                "out_of_range_rows": out_of_range,
                "eligible_for_training": eligible,
            })
            if valid.size >= MIN_TARGET_ROWS and out_of_range > 0:
                blocking_messages.append(f"{well_alias}/{target}: target values remain outside [0, 1].")
        for feature in MODEL_FEATURE_CANDIDATES:
            coverage = float(frame[feature].notna().mean()) if feature in frame else 0.0
            feature_rows.append({
                "well_alias": well_alias,
                "feature": feature,
                "coverage": coverage,
                "available": bool(coverage >= MIN_FEATURE_COVERAGE),
                "minimum": float(frame[feature].min()) if feature in frame and frame[feature].notna().any() else np.nan,
                "maximum": float(frame[feature].max()) if feature in frame and frame[feature].notna().any() else np.nan,
            })

    target_audit = pd.DataFrame(target_rows)
    readiness = pd.DataFrame(readiness_rows)
    feature_coverage = pd.DataFrame(feature_rows)
    target_audit.to_csv(AUDIT_ROOT / "target_audit.csv", index=False)
    readiness.to_csv(AUDIT_ROOT / "well_readiness_audit.csv", index=False)
    feature_coverage.to_csv(AUDIT_ROOT / "feature_coverage_audit.csv", index=False)

    leakage_rows: list[dict[str, Any]] = []
    for target in TARGET_COLUMNS:
        for feature in MODEL_FEATURE_CANDIDATES:
            leakage_rows.append({
                "target": target,
                "candidate_feature": feature,
                "decision": "allowed",
                "reason": "Canonical measured/derived feature; target and depth columns are separately excluded.",
            })
        leakage_rows.extend([
            {"target": target, "candidate_feature": other_target, "decision": "excluded", "reason": "target_or_target_like_leakage"}
            for other_target in TARGET_COLUMNS
        ])
        leakage_rows.extend([
            {"target": target, "candidate_feature": field, "decision": "excluded", "reason": "identifier_alignment_or_context"}
            for field in ("well_alias", "source_workbook", "source_sheet", "source_row", "depth_original", "depth_m")
        ])
    leakage = pd.DataFrame(leakage_rows)
    leakage.to_csv(AUDIT_ROOT / "target_leakage_policy_audit.csv", index=False)

    if blocking_messages:
        (AUDIT_ROOT / "blocking_issues.txt").write_text("\n".join(blocking_messages), encoding="utf-8")
        raise RuntimeError("Standardization audit blocked modeling:\n- " + "\n- ".join(blocking_messages))
    return target_audit, readiness, feature_coverage, leakage

## 4. Cross-well model functions

The default random-forest pipeline uses training-fold median imputation and training-fold min–max scaling. It creates single-source transfer models, leave-one-well-out models, prediction intervals, baseline-relative skill metrics, feature-importance summaries, and feature-range-shift audits.

In [ ]:
# =============================================================================
# Modeling helpers
# =============================================================================

def select_common_features(
    well_frames: dict[str, pd.DataFrame],
    required_wells: list[str],
    training_wells: list[str],
) -> tuple[list[str], pd.DataFrame]:
    rows: list[dict[str, Any]] = []
    selected: list[str] = []
    for feature in MODEL_FEATURE_CANDIDATES:
        coverage_by_well = {
            well: float(well_frames[well][feature].notna().mean()) if feature in well_frames[well] else 0.0
            for well in required_wells
        }
        training_parts = [
            well_frames[well][feature] for well in training_wells if feature in well_frames[well]
        ]
        training_values = pd.concat(training_parts, ignore_index=True) if training_parts else pd.Series(dtype=float)
        training_non_null = training_values.dropna()
        reason = ""
        if any(value < MIN_FEATURE_COVERAGE for value in coverage_by_well.values()):
            reason = "insufficient_coverage_in_at_least_one_required_well"
        elif training_non_null.nunique() <= 1:
            reason = "constant_or_empty_in_training_wells"
        else:
            selected.append(feature)
        row = {
            "feature": feature,
            "decision": "included" if not reason else "excluded",
            "reason": reason,
            "training_non_null_rows": int(training_non_null.size),
            "training_unique_values": int(training_non_null.nunique()),
        }
        row.update({f"coverage_{well}": value for well, value in coverage_by_well.items()})
        rows.append(row)
    if USE_DEPTH_AS_FEATURE:
        selected.append("depth_m")
    if len(selected) < 3:
        raise RuntimeError(
            f"Only {len(selected)} common features were available for wells {required_wells}. "
            "Review column mappings and feature coverage before modeling."
        )
    return selected, pd.DataFrame(rows)


def make_model() -> Pipeline:
    steps: list[tuple[str, Any]] = [("imputer", SimpleImputer(strategy="median"))]
    if SCALE_MODEL_FEATURES or MODEL_KIND == "mlp":
        steps.append(("scaler", MinMaxScaler()))
    if MODEL_KIND == "random_forest":
        steps.append(("model", RandomForestRegressor(
            n_estimators=400,
            min_samples_leaf=3,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )))
    elif MODEL_KIND == "mlp":
        steps.append(("model", MLPRegressor(
            hidden_layer_sizes=(64, 32),
            max_iter=1000,
            early_stopping=True,
            random_state=RANDOM_STATE,
        )))
    else:
        raise ValueError(f"Unsupported MODEL_KIND: {MODEL_KIND}")
    return Pipeline(steps)


def metric_record(y_true: pd.Series, y_pred: np.ndarray, train_mean: float) -> dict[str, Any]:
    true = pd.to_numeric(y_true, errors="coerce").to_numpy(dtype=float)
    pred = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(true) & np.isfinite(pred)
    true = true[valid]
    pred = pred[valid]
    if len(true) == 0:
        return {
            "scored_rows": 0, "mae": np.nan, "rmse": np.nan, "r2": np.nan,
            "bias": np.nan, "baseline_mae": np.nan, "baseline_rmse": np.nan,
            "rmse_skill_vs_train_mean": np.nan,
        }
    baseline = np.full_like(true, train_mean, dtype=float)
    rmse = float(np.sqrt(mean_squared_error(true, pred)))
    baseline_rmse = float(np.sqrt(mean_squared_error(true, baseline)))
    return {
        "scored_rows": int(len(true)),
        "mae": float(mean_absolute_error(true, pred)),
        "rmse": rmse,
        "r2": float(r2_score(true, pred)) if len(true) >= 2 and np.nanstd(true) > 0 else np.nan,
        "bias": float(np.mean(pred - true)),
        "baseline_mae": float(mean_absolute_error(true, baseline)),
        "baseline_rmse": baseline_rmse,
        "rmse_skill_vs_train_mean": float(1.0 - rmse / baseline_rmse) if baseline_rmse > 0 else np.nan,
    }


def forest_prediction_interval(model: Pipeline, X: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    final_model = model.named_steps.get("model")
    if not hasattr(final_model, "estimators_"):
        nan = np.full(len(X), np.nan)
        return nan, nan
    transformed = model.named_steps["imputer"].transform(X)
    if "scaler" in model.named_steps:
        transformed = model.named_steps["scaler"].transform(transformed)
    tree_predictions = np.vstack([tree.predict(transformed) for tree in final_model.estimators_])
    return np.quantile(tree_predictions, 0.10, axis=0), np.quantile(tree_predictions, 0.90, axis=0)


def feature_importance_records(model: Pipeline, feature_columns: list[str]) -> list[dict[str, Any]]:
    final_model = model.named_steps.get("model")
    if hasattr(final_model, "feature_importances_"):
        return [
            {"feature": feature, "importance": float(importance), "importance_type": "random_forest_impurity"}
            for feature, importance in zip(feature_columns, final_model.feature_importances_)
        ]
    return [
        {"feature": feature, "importance": np.nan, "importance_type": "not_available_for_model_kind"}
        for feature in feature_columns
    ]


def feature_shift_records(
    train_frame: pd.DataFrame,
    test_frame: pd.DataFrame,
    feature_columns: list[str],
    target: str,
    model_run: str,
    test_well: str,
) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for feature in feature_columns:
        train_values = train_frame[feature].dropna()
        test_values = test_frame[feature].dropna()
        if train_values.empty or test_values.empty:
            outside_fraction = np.nan
            train_min = train_max = np.nan
        else:
            train_min = float(train_values.min())
            train_max = float(train_values.max())
            outside_fraction = float(((test_values < train_min) | (test_values > train_max)).mean())
        rows.append({
            "model_run": model_run,
            "target": target,
            "test_well": test_well,
            "feature": feature,
            "train_min": train_min,
            "train_max": train_max,
            "test_min": float(test_values.min()) if not test_values.empty else np.nan,
            "test_max": float(test_values.max()) if not test_values.empty else np.nan,
            "test_fraction_outside_training_range": outside_fraction,
        })
    return rows


def _target_is_trainable(frame: pd.DataFrame, target: str) -> bool:
    if target not in frame:
        return False
    valid = frame[target].dropna()
    return bool(len(valid) >= MIN_TARGET_ROWS and valid.nunique() > 1)


def fit_single_training_well_transfer(
    well_frames: dict[str, pd.DataFrame],
    train_well: str = SINGLE_TRAIN_WELL,
) -> dict[str, pd.DataFrame]:
    if train_well not in well_frames:
        raise ValueError(f"Unknown train well: {train_well}")

    prediction_frames: list[pd.DataFrame] = []
    metric_rows: list[dict[str, Any]] = []
    importance_rows: list[dict[str, Any]] = []
    feature_policy_frames: list[pd.DataFrame] = []
    shift_rows: list[dict[str, Any]] = []
    model_inventory: list[dict[str, Any]] = []

    for target in TARGET_COLUMNS:
        train_frame = well_frames[train_well]
        if not _target_is_trainable(train_frame, target):
            model_inventory.append({
                "model_run": "single_training_well_transfer",
                "target": target,
                "train_wells": train_well,
                "status": "blocked",
                "reason": "training well lacks enough nonconstant target rows",
            })
            continue

        required_wells = list(well_frames)
        feature_columns, feature_policy = select_common_features(well_frames, required_wells, [train_well])
        feature_policy.insert(0, "target", target)
        feature_policy.insert(0, "model_run", "single_training_well_transfer")
        feature_policy_frames.append(feature_policy)

        train_mask = train_frame[target].notna() & train_frame[feature_columns].notna().any(axis=1)
        X_train = train_frame.loc[train_mask, feature_columns].copy()
        y_train = train_frame.loc[train_mask, target].copy()
        model = make_model()
        model.fit(X_train, y_train)
        train_mean = float(y_train.mean())

        model_dir = MODEL_RUN_ROOT / "single_training_well_transfer" / target
        model_dir.mkdir(parents=True, exist_ok=True)
        model_path = model_dir / f"{train_well}_{target}.joblib"
        joblib.dump({
            "model": model,
            "feature_columns": feature_columns,
            "target": target,
            "train_wells": [train_well],
            "well_alias_policy": "WellA/WellB/WellC/WellD",
        }, model_path)

        for record in feature_importance_records(model, feature_columns):
            record.update({
                "model_run": "single_training_well_transfer",
                "target": target,
                "train_wells": train_well,
            })
            importance_rows.append(record)

        for well_alias, frame in well_frames.items():
            score_mask = frame[feature_columns].notna().any(axis=1)
            X_score = frame.loc[score_mask, feature_columns].copy()
            if X_score.empty:
                continue
            y_pred_raw = model.predict(X_score)
            y_pred = np.clip(y_pred_raw, 0.0, 1.0) if CLIP_SATURATION_PREDICTIONS else y_pred_raw
            p10, p90 = forest_prediction_interval(model, X_score)
            if CLIP_SATURATION_PREDICTIONS:
                p10 = np.clip(p10, 0.0, 1.0)
                p90 = np.clip(p90, 0.0, 1.0)
            prediction = pd.DataFrame({
                "model_run": "single_training_well_transfer",
                "target": target,
                "train_wells": train_well,
                "well_alias": well_alias,
                "source_row": frame.loc[score_mask, "source_row"].to_numpy(),
                "depth_original": frame.loc[score_mask, "depth_original"].to_numpy(),
                "depth_original_unit": frame.loc[score_mask, "depth_original_unit"].to_numpy(),
                "depth_m": frame.loc[score_mask, "depth_m"].to_numpy(),
                "y_pred_raw": y_pred_raw,
                "y_pred": y_pred,
                "y_pred_p10": p10,
                "y_pred_p90": p90,
                "evaluation_role": "training_resubstitution" if well_alias == train_well else "heldout_well",
            })
            if target in frame:
                prediction["y_true"] = frame.loc[score_mask, target].to_numpy()
            else:
                prediction["y_true"] = np.nan
            prediction_frames.append(prediction)

            valid_target = prediction["y_true"].notna()
            metrics = metric_record(
                prediction.loc[valid_target, "y_true"],
                prediction.loc[valid_target, "y_pred"].to_numpy(),
                train_mean,
            )
            metric_rows.append({
                "model_run": "single_training_well_transfer",
                "target": target,
                "train_wells": train_well,
                "test_well": well_alias,
                "evaluation_role": "training_resubstitution" if well_alias == train_well else "heldout_well",
                "feature_count": len(feature_columns),
                "train_rows": len(X_train),
                **metrics,
            })
            shift_rows.extend(feature_shift_records(
                train_frame.loc[train_mask], frame.loc[score_mask], feature_columns,
                target, "single_training_well_transfer", well_alias,
            ))

        model_inventory.append({
            "model_run": "single_training_well_transfer",
            "target": target,
            "train_wells": train_well,
            "status": "trained",
            "feature_count": len(feature_columns),
            "train_rows": len(X_train),
            "model_path": str(model_path),
        })

    predictions = pd.concat(prediction_frames, ignore_index=True, sort=False) if prediction_frames else pd.DataFrame()
    metrics = pd.DataFrame(metric_rows)
    importance = pd.DataFrame(importance_rows)
    feature_policy = pd.concat(feature_policy_frames, ignore_index=True, sort=False) if feature_policy_frames else pd.DataFrame()
    shift = pd.DataFrame(shift_rows)
    inventory = pd.DataFrame(model_inventory)

    out_dir = PREDICTION_ROOT / "single_training_well_transfer"
    out_dir.mkdir(parents=True, exist_ok=True)
    if not predictions.empty:
        for (target, well_alias), group in predictions.groupby(["target", "well_alias"]):
            group.to_csv(out_dir / f"{train_well}_to_{well_alias}_{target}_predictions.csv", index=False)
    metrics.to_csv(METRIC_ROOT / "single_training_well_transfer_metrics.csv", index=False)
    importance.to_csv(METRIC_ROOT / "single_training_well_feature_importance.csv", index=False)
    feature_policy.to_csv(AUDIT_ROOT / "single_training_well_feature_policy.csv", index=False)
    shift.to_csv(AUDIT_ROOT / "single_training_well_feature_shift.csv", index=False)
    inventory.to_csv(METRIC_ROOT / "single_training_well_model_inventory.csv", index=False)
    return {
        "predictions": predictions,
        "metrics": metrics,
        "feature_importance": importance,
        "feature_policy": feature_policy,
        "feature_shift": shift,
        "model_inventory": inventory,
    }


def fit_leave_one_well_out(well_frames: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    prediction_frames: list[pd.DataFrame] = []
    metric_rows: list[dict[str, Any]] = []
    importance_rows: list[dict[str, Any]] = []
    feature_policy_frames: list[pd.DataFrame] = []
    shift_rows: list[dict[str, Any]] = []
    inventory_rows: list[dict[str, Any]] = []

    for target in TARGET_COLUMNS:
        labeled_wells = [well for well, frame in well_frames.items() if _target_is_trainable(frame, target)]
        if len(labeled_wells) < 2:
            inventory_rows.append({
                "model_run": "leave_one_well_out",
                "target": target,
                "status": "blocked",
                "reason": "fewer than two wells have enough nonconstant reference target rows",
            })
            continue

        for heldout_well in labeled_wells:
            training_wells = [well for well in labeled_wells if well != heldout_well]
            required_wells = training_wells + [heldout_well]
            feature_columns, feature_policy = select_common_features(well_frames, required_wells, training_wells)
            feature_policy.insert(0, "heldout_well", heldout_well)
            feature_policy.insert(0, "target", target)
            feature_policy.insert(0, "model_run", "leave_one_well_out")
            feature_policy_frames.append(feature_policy)

            train_parts: list[pd.DataFrame] = []
            for well in training_wells:
                frame = well_frames[well]
                mask = frame[target].notna() & frame[feature_columns].notna().any(axis=1)
                part = frame.loc[mask, feature_columns + [target]].copy()
                part["well_alias"] = well
                train_parts.append(part)
            train_data = pd.concat(train_parts, ignore_index=True, sort=False)
            X_train = train_data[feature_columns]
            y_train = train_data[target]

            test_frame = well_frames[heldout_well]
            score_mask = test_frame[feature_columns].notna().any(axis=1)
            X_test = test_frame.loc[score_mask, feature_columns].copy()
            if X_test.empty:
                inventory_rows.append({
                    "model_run": "leave_one_well_out",
                    "target": target,
                    "heldout_well": heldout_well,
                    "train_wells": ",".join(training_wells),
                    "status": "blocked",
                    "reason": "held-out well has no scoreable feature rows",
                })
                continue

            model = make_model()
            model.fit(X_train, y_train)
            y_pred_raw = model.predict(X_test)
            y_pred = np.clip(y_pred_raw, 0.0, 1.0) if CLIP_SATURATION_PREDICTIONS else y_pred_raw
            p10, p90 = forest_prediction_interval(model, X_test)
            if CLIP_SATURATION_PREDICTIONS:
                p10 = np.clip(p10, 0.0, 1.0)
                p90 = np.clip(p90, 0.0, 1.0)
            train_mean = float(y_train.mean())

            model_dir = MODEL_RUN_ROOT / "leave_one_well_out" / target / f"heldout_{heldout_well}"
            model_dir.mkdir(parents=True, exist_ok=True)
            model_path = model_dir / "model.joblib"
            joblib.dump({
                "model": model,
                "feature_columns": feature_columns,
                "target": target,
                "train_wells": training_wells,
                "heldout_well": heldout_well,
            }, model_path)

            prediction = pd.DataFrame({
                "model_run": "leave_one_well_out",
                "target": target,
                "train_wells": ",".join(training_wells),
                "well_alias": heldout_well,
                "source_row": test_frame.loc[score_mask, "source_row"].to_numpy(),
                "depth_original": test_frame.loc[score_mask, "depth_original"].to_numpy(),
                "depth_original_unit": test_frame.loc[score_mask, "depth_original_unit"].to_numpy(),
                "depth_m": test_frame.loc[score_mask, "depth_m"].to_numpy(),
                "y_true": test_frame.loc[score_mask, target].to_numpy(),
                "y_pred_raw": y_pred_raw,
                "y_pred": y_pred,
                "y_pred_p10": p10,
                "y_pred_p90": p90,
                "evaluation_role": "heldout_well",
            })
            prediction_frames.append(prediction)
            valid_target = prediction["y_true"].notna()
            metrics = metric_record(
                prediction.loc[valid_target, "y_true"],
                prediction.loc[valid_target, "y_pred"].to_numpy(),
                train_mean,
            )
            metric_rows.append({
                "model_run": "leave_one_well_out",
                "target": target,
                "train_wells": ",".join(training_wells),
                "test_well": heldout_well,
                "evaluation_role": "heldout_well",
                "feature_count": len(feature_columns),
                "train_rows": len(X_train),
                **metrics,
            })
            for record in feature_importance_records(model, feature_columns):
                record.update({
                    "model_run": "leave_one_well_out",
                    "target": target,
                    "train_wells": ",".join(training_wells),
                    "heldout_well": heldout_well,
                })
                importance_rows.append(record)
            shift_rows.extend(feature_shift_records(
                train_data, test_frame.loc[score_mask], feature_columns,
                target, "leave_one_well_out", heldout_well,
            ))
            inventory_rows.append({
                "model_run": "leave_one_well_out",
                "target": target,
                "heldout_well": heldout_well,
                "train_wells": ",".join(training_wells),
                "status": "trained",
                "feature_count": len(feature_columns),
                "train_rows": len(X_train),
                "test_rows": len(X_test),
                "model_path": str(model_path),
            })

    predictions = pd.concat(prediction_frames, ignore_index=True, sort=False) if prediction_frames else pd.DataFrame()
    metrics = pd.DataFrame(metric_rows)
    importance = pd.DataFrame(importance_rows)
    feature_policy = pd.concat(feature_policy_frames, ignore_index=True, sort=False) if feature_policy_frames else pd.DataFrame()
    shift = pd.DataFrame(shift_rows)
    inventory = pd.DataFrame(inventory_rows)

    out_dir = PREDICTION_ROOT / "leave_one_well_out"
    out_dir.mkdir(parents=True, exist_ok=True)
    if not predictions.empty:
        for (target, well_alias), group in predictions.groupby(["target", "well_alias"]):
            group.to_csv(out_dir / f"heldout_{well_alias}_{target}_predictions.csv", index=False)
    metrics.to_csv(METRIC_ROOT / "leave_one_well_out_metrics.csv", index=False)
    importance.to_csv(METRIC_ROOT / "leave_one_well_out_feature_importance.csv", index=False)
    feature_policy.to_csv(AUDIT_ROOT / "leave_one_well_out_feature_policy.csv", index=False)
    shift.to_csv(AUDIT_ROOT / "leave_one_well_out_feature_shift.csv", index=False)
    inventory.to_csv(METRIC_ROOT / "leave_one_well_out_model_inventory.csv", index=False)
    return {
        "predictions": predictions,
        "metrics": metrics,
        "feature_importance": importance,
        "feature_policy": feature_policy,
        "feature_shift": shift,
        "model_inventory": inventory,
    }

## 5. Slide-ready figures, summary tables, and validation bundle

Full predictions and fitted models stay local. The validation ZIP contains only summary audits, metrics, manifests, and figures.

In [ ]:
# =============================================================================
# Figures and summary exports
# =============================================================================

def _save_figure(path: Path, manifest: list[dict[str, Any]], source: str, figure_type: str) -> None:
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.close()
    manifest.append({"figure": str(path), "source": source, "figure_type": figure_type})


def generate_model_figures(
    single_results: dict[str, pd.DataFrame],
    loow_results: dict[str, pd.DataFrame],
) -> pd.DataFrame:
    manifest: list[dict[str, Any]] = []

    single_predictions = single_results.get("predictions", pd.DataFrame())
    single_metrics = single_results.get("metrics", pd.DataFrame())
    loow_predictions = loow_results.get("predictions", pd.DataFrame())
    loow_metrics = loow_results.get("metrics", pd.DataFrame())

    if not loow_predictions.empty:
        for target, target_df in loow_predictions.groupby("target"):
            plot_df = target_df.dropna(subset=["y_true", "y_pred"]).copy()
            if not plot_df.empty:
                fig, ax = plt.subplots(figsize=(7, 7))
                for well_alias, well_df in plot_df.groupby("well_alias"):
                    ax.scatter(well_df["y_true"], well_df["y_pred"], s=12, alpha=0.55, label=well_alias)
                lo = float(min(plot_df["y_true"].min(), plot_df["y_pred"].min()))
                hi = float(max(plot_df["y_true"].max(), plot_df["y_pred"].max()))
                ax.plot([lo, hi], [lo, hi], linewidth=1)
                ax.set_title(f"Held-out well predictions: {target}")
                ax.set_xlabel("Reference target")
                ax.set_ylabel("Predicted target")
                ax.legend()
                _save_figure(
                    PAPER_EXPORT_ROOT / f"figure_loow_predicted_vs_reference_{target}.png",
                    manifest,
                    "leave_one_well_out_predictions",
                    "heldout_predicted_vs_reference",
                )

                plot_df["residual"] = plot_df["y_pred"] - plot_df["y_true"]
                fig, ax = plt.subplots(figsize=(8, 5))
                ax.hist(plot_df["residual"].dropna(), bins=35)
                ax.set_title(f"Held-out residuals: {target}")
                ax.set_xlabel("Residual = predicted - reference")
                ax.set_ylabel("Rows")
                _save_figure(
                    PAPER_EXPORT_ROOT / f"figure_loow_residual_histogram_{target}.png",
                    manifest,
                    "leave_one_well_out_predictions",
                    "heldout_residual_histogram",
                )

            profile_df = target_df.dropna(subset=["depth_m", "y_pred"]).copy()
            if not profile_df.empty:
                fig, ax = plt.subplots(figsize=(8, 7))
                for well_alias, well_df in profile_df.groupby("well_alias"):
                    ordered = well_df.sort_values("depth_m")
                    ax.plot(ordered["y_pred"], ordered["depth_m"], label=well_alias, linewidth=1)
                ax.invert_yaxis()
                ax.set_title(f"Held-out prediction by depth: {target}")
                ax.set_xlabel("Predicted target")
                ax.set_ylabel("Depth (m)")
                ax.legend()
                _save_figure(
                    PAPER_EXPORT_ROOT / f"figure_loow_prediction_by_depth_{target}.png",
                    manifest,
                    "leave_one_well_out_predictions",
                    "heldout_prediction_depth_profile",
                )

    if not loow_metrics.empty:
        for target, target_df in loow_metrics.groupby("target"):
            valid = target_df[target_df["scored_rows"] > 0].copy()
            if valid.empty:
                continue
            for metric in ("mae", "rmse", "r2", "rmse_skill_vs_train_mean"):
                if metric not in valid or valid[metric].dropna().empty:
                    continue
                fig, ax = plt.subplots(figsize=(8, 5))
                ax.bar(valid["test_well"], valid[metric])
                ax.set_title(f"Held-out {metric}: {target}")
                ax.set_xlabel("Held-out well")
                ax.set_ylabel(metric)
                _save_figure(
                    PAPER_EXPORT_ROOT / f"figure_loow_{metric}_{target}.png",
                    manifest,
                    "leave_one_well_out_metrics",
                    f"heldout_metric_{metric}",
                )

    if not single_predictions.empty:
        heldout = single_predictions[single_predictions["evaluation_role"] == "heldout_well"].copy()
        for target, target_df in heldout.groupby("target"):
            profile_df = target_df.dropna(subset=["depth_m", "y_pred"])
            if profile_df.empty:
                continue
            fig, ax = plt.subplots(figsize=(8, 7))
            for well_alias, well_df in profile_df.groupby("well_alias"):
                ordered = well_df.sort_values("depth_m")
                ax.plot(ordered["y_pred"], ordered["depth_m"], label=well_alias, linewidth=1)
            ax.invert_yaxis()
            ax.set_title(f"{SINGLE_TRAIN_WELL} transfer prediction by depth: {target}")
            ax.set_xlabel("Predicted target")
            ax.set_ylabel("Depth (m)")
            ax.legend()
            _save_figure(
                PAPER_EXPORT_ROOT / f"figure_{SINGLE_TRAIN_WELL}_transfer_by_depth_{target}.png",
                manifest,
                "single_training_well_transfer_predictions",
                "single_training_well_transfer_depth_profile",
            )

    importance_parts = []
    for source_name, results in (("single", single_results), ("loow", loow_results)):
        frame = results.get("feature_importance", pd.DataFrame())
        if not frame.empty:
            temp = frame.copy()
            temp["source_run"] = source_name
            importance_parts.append(temp)
    if importance_parts:
        importance = pd.concat(importance_parts, ignore_index=True, sort=False)
        valid = importance.dropna(subset=["importance"])
        if not valid.empty:
            for target, target_df in valid.groupby("target"):
                aggregated = target_df.groupby("feature", as_index=False)["importance"].mean().sort_values("importance", ascending=False).head(15)
                fig, ax = plt.subplots(figsize=(10, 5))
                ax.bar(aggregated["feature"], aggregated["importance"])
                ax.set_title(f"Mean feature importance: {target}")
                ax.set_xlabel("Feature")
                ax.set_ylabel("Importance")
                ax.tick_params(axis="x", rotation=45)
                _save_figure(
                    PAPER_EXPORT_ROOT / f"figure_feature_importance_{target}.png",
                    manifest,
                    "model_feature_importance",
                    "feature_importance",
                )

    manifest_df = pd.DataFrame(manifest)
    manifest_df.to_csv(PAPER_EXPORT_ROOT / "figure_manifest.csv", index=False)
    return manifest_df


def export_summary_tables(
    audits: dict[str, pd.DataFrame],
    target_audit: pd.DataFrame,
    readiness: pd.DataFrame,
    feature_coverage: pd.DataFrame,
    leakage: pd.DataFrame,
    single_results: dict[str, pd.DataFrame],
    loow_results: dict[str, pd.DataFrame],
    figure_manifest: pd.DataFrame,
) -> dict[str, Path]:
    outputs: dict[str, Path] = {}
    tables = {
        "source_layout_inventory.csv": audits["source_layout"],
        "column_mapping_audit.csv": audits["column_mapping"],
        "unit_conversion_audit.csv": audits["unit_conversion"],
        "refined_sheet_inventory.csv": audits["refined_sheet_inventory"],
        "ignored_input_inventory.csv": audits["ignored_inputs"],
        "target_audit.csv": target_audit,
        "well_readiness_audit.csv": readiness,
        "feature_coverage_audit.csv": feature_coverage,
        "target_leakage_policy_audit.csv": leakage,
        "single_training_well_transfer_metrics.csv": single_results.get("metrics", pd.DataFrame()),
        "leave_one_well_out_metrics.csv": loow_results.get("metrics", pd.DataFrame()),
        "single_training_well_model_inventory.csv": single_results.get("model_inventory", pd.DataFrame()),
        "leave_one_well_out_model_inventory.csv": loow_results.get("model_inventory", pd.DataFrame()),
        "single_training_well_feature_importance.csv": single_results.get("feature_importance", pd.DataFrame()),
        "leave_one_well_out_feature_importance.csv": loow_results.get("feature_importance", pd.DataFrame()),
        "figure_manifest.csv": figure_manifest,
    }
    for filename, frame in tables.items():
        path = PAPER_EXPORT_ROOT / filename
        frame.to_csv(path, index=False)
        outputs[filename] = path

    manifest = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "data_directory": str(DATA_DIR),
        "source_files": [spec["filename"] for spec in SOURCE_SPECS],
        "well_mapping": {spec["well_alias"]: {"filename": spec["filename"], "sheet": spec["sheet_name"]} for spec in SOURCE_SPECS},
        "curated_files_policy": "ignored",
        "refined_sheet_policy": "alignment_qc_only",
        "single_training_well": SINGLE_TRAIN_WELL,
        "model_kind": MODEL_KIND,
        "model_feature_scaling": SCALE_MODEL_FEATURES,
        "allow_nmr_porosity_as_feature": ALLOW_NMR_POROSITY_AS_FEATURE,
        "clip_saturation_predictions": CLIP_SATURATION_PREDICTIONS,
        "depth_feature_enabled": USE_DEPTH_AS_FEATURE,
        "outputs": {name: str(path) for name, path in outputs.items()},
        "claims_boundary": "Training-resubstitution metrics are not validation. Leave-one-well-out metrics are the primary cross-well validation evidence.",
    }
    manifest_path = PAPER_EXPORT_ROOT / "paper_slide_deliverable_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    outputs[manifest_path.name] = manifest_path
    return outputs


def build_validation_bundle() -> Path:
    if VALIDATION_BUNDLE_ROOT.exists():
        shutil.rmtree(VALIDATION_BUNDLE_ROOT)
    VALIDATION_BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)

    safe_files = [
        *PAPER_EXPORT_ROOT.glob("*.csv"),
        *PAPER_EXPORT_ROOT.glob("*.json"),
        *PAPER_EXPORT_ROOT.glob("*.png"),
    ]
    for source in safe_files:
        destination = VALIDATION_BUNDLE_ROOT / source.name
        shutil.copy2(source, destination)

    readme = VALIDATION_BUNDLE_ROOT / "README.txt"
    readme.write_text(
        "North Slope validation bundle\n\n"
        "Contains summary tables, audits, metrics, and figures only.\n"
        "Does not contain source workbook rows, full prediction CSVs, or fitted model files.\n"
        "Use leave-one-well-out metrics for cross-well validation; training-resubstitution metrics are diagnostic only.\n",
        encoding="utf-8",
    )
    archive_path = shutil.make_archive(str(VALIDATION_BUNDLE_ROOT), "zip", root_dir=VALIDATION_BUNDLE_ROOT)
    return Path(archive_path)


def run_full_pipeline(data_dir: Path | None = None) -> dict[str, Any]:
    global DATA_DIR
    if data_dir is not None:
        DATA_DIR = Path(data_dir)
    well_frames, audits = standardize_all_sources(DATA_DIR)
    target_audit, readiness, feature_coverage, leakage = build_target_and_readiness_audits(well_frames)
    single_results = fit_single_training_well_transfer(well_frames, SINGLE_TRAIN_WELL)
    loow_results = fit_leave_one_well_out(well_frames)
    figure_manifest = generate_model_figures(single_results, loow_results)
    summary_outputs = export_summary_tables(
        audits, target_audit, readiness, feature_coverage, leakage,
        single_results, loow_results, figure_manifest,
    )
    bundle = build_validation_bundle()
    return {
        "well_frames": well_frames,
        "audits": audits,
        "target_audit": target_audit,
        "readiness": readiness,
        "feature_coverage": feature_coverage,
        "leakage": leakage,
        "single_results": single_results,
        "loow_results": loow_results,
        "figure_manifest": figure_manifest,
        "summary_outputs": summary_outputs,
        "validation_bundle": bundle,
    }

## 6. Preflight: rebuild and audit the four standardized wells

Run this cell before interpreting any model output. A valid preflight should show four wells, physical depth ranges, hydrate targets for all four wells, and water targets for the wells where they are actually supplied.

In [ ]:
print("Data folder:", DATA_DIR)
print("\nRequired original workbooks:")
for spec in SOURCE_SPECS:
    path = DATA_DIR / spec["filename"]
    print(f"  {spec['well_alias']}: {spec['filename']} ->", "FOUND" if path.exists() else "MISSING")
print("\nCurated files are ignored:")
for path in sorted(DATA_DIR.glob(CURATED_GLOB)):
    print(" ", path.name)

well_frames, audits = standardize_all_sources(DATA_DIR)
target_audit, readiness, feature_coverage, leakage = build_target_and_readiness_audits(well_frames)

print("\nSTANDARDIZED WELL SHAPES")
for well_alias, frame in well_frames.items():
    print(
        well_alias,
        frame.shape,
        "original depth:",
        round(float(frame["depth_original"].min()), 3),
        "to",
        round(float(frame["depth_original"].max()), 3),
        str(frame["depth_original_unit"].dropna().iloc[0]),
    )

print("\nSOURCE LAYOUT")
display(audits["source_layout"])
print("\nTARGET AUDIT")
display(target_audit)
print("\nWELL READINESS")
display(readiness)
print("\nFEATURE COVERAGE")
display(feature_coverage.pivot(index="feature", columns="well_alias", values="coverage"))
print("\nStandardized tables:", STANDARDIZED_ROOT)

## 7. Run cross-well models and build presentation exports

Leave-one-well-out metrics are the primary validation evidence. Training-resubstitution metrics are diagnostic only and must not be presented as held-out performance.

In [ ]:
single_results = fit_single_training_well_transfer(well_frames, SINGLE_TRAIN_WELL)
loow_results = fit_leave_one_well_out(well_frames)
figure_manifest = generate_model_figures(single_results, loow_results)
summary_outputs = export_summary_tables(
    audits,
    target_audit,
    readiness,
    feature_coverage,
    leakage,
    single_results,
    loow_results,
    figure_manifest,
)
validation_bundle = build_validation_bundle()

results = {
    "well_frames": well_frames,
    "audits": audits,
    "target_audit": target_audit,
    "readiness": readiness,
    "feature_coverage": feature_coverage,
    "leakage": leakage,
    "single_results": single_results,
    "loow_results": loow_results,
    "figure_manifest": figure_manifest,
    "summary_outputs": summary_outputs,
    "validation_bundle": validation_bundle,
}

print("\nSINGLE-TRAINING-WELL TRANSFER METRICS")
display(single_results["metrics"])
print("\nLEAVE-ONE-WELL-OUT METRICS — PRIMARY CROSS-WELL VALIDATION")
display(loow_results["metrics"])
print("\nMODEL INVENTORY")
display(pd.concat([
    single_results["model_inventory"],
    loow_results["model_inventory"],
], ignore_index=True, sort=False))
print("\nFIGURE MANIFEST")
display(figure_manifest)

print("\nPAPER/SLIDE EXPORT FOLDER:", PAPER_EXPORT_ROOT)
print("VALIDATION BUNDLE:", validation_bundle)
print("STANDARDIZED WELL FOLDER:", STANDARDIZED_ROOT)
print("FULL LOCAL PREDICTIONS:", PREDICTION_ROOT)
print("LOCAL FITTED MODELS:", MODEL_RUN_ROOT)

## Output interpretation

Use:

```text
outputs_runtime/paper_slide_model_exports/leave_one_well_out_metrics.csv
```

for cross-well performance review. Do not use training-resubstitution R² as validation. Review the target, unit, leakage, feature-coverage, and feature-shift audits before using any figure in the manuscript or slide deck.

The summary-only review file is:

```text
north_slope_validation_bundle.zip
```

It intentionally excludes source workbook rows, full prediction tables, and fitted models.